# 03 — Match parents to children in SQL and estimate the IGE

**Reads:** `data/processed/psid.sqlite`. **Writes:** `data/processed/matched.csv` (one row per child with parent
and child income measures), `results/tables/ige_estimates.csv`, `results/tables/ige_by_child_age.csv`.

All matching and aggregation happens in SQL run through `sqlite3`; pandas only receives results.
Every query in the post is copied from here, so write them as named strings (or `.sql` files in `code/sql/`).

In [1]:
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd
import statsmodels.api as sm

ROOT = Path.cwd().parent
PROC, TAB = ROOT / "data" / "processed", ROOT / "results" / "tables"
conn = sqlite3.connect(PROC / "psid.sqlite")

def q(sql, **params):
    return pd.read_sql_query(sql, conn, params=params)

def make_view(name, sql):
    """(Re)create a temporary view: it lives only for this connection, the database file is untouched."""
    conn.execute(f"DROP VIEW IF EXISTS {name}")
    conn.execute(sql)

## 0. Indexes
Without an index, a join compares every row with every row: the child → father self-join below is
85,536 × 85,536 ≈ 7 billion comparisons. An index is a sorted lookup on the join columns, so each match is found
directly. One index per key used in a join; `IF NOT EXISTS` makes the cell safe to re-run.

In [2]:
INDEX_SQL = [
    "CREATE INDEX IF NOT EXISTS ix_individuals_id ON individuals(id_1968, person_num)",
    "CREATE INDEX IF NOT EXISTS ix_persons_id ON persons(id_1968, person_num, year)",
    "CREATE INDEX IF NOT EXISTS ix_persons_interview ON persons(year, interview)",
    "CREATE INDEX IF NOT EXISTS ix_families_key ON families(year, fam_interview)",
]
for sql in INDEX_SQL:
    conn.execute(sql)
conn.commit()

## 1. SELECT / WHERE — the child sample
Define who counts as a "child" observation: a person-year in which the person is head or spouse of their own
family unit (`rel_head`), within the age window you chose. Count them.

In [3]:
CHILD_N_SQL = """
SELECT COUNT(*)
FROM persons
WHERE (rel_head IN (1,2) OR rel_head IN (10, 20, 22))
    AND age BETWEEN 25 AND 45
"""
q(CHILD_N_SQL)

,COUNT(*)
0,259398


In [4]:
CHILD_COUNT_SQL = """
SELECT COUNT(*) FROM(
    SELECT DISTINCT id_1968, person_num
    FROM persons
    WHERE (rel_head IN (1, 2) OR rel_head IN (10, 20, 22))
        AND age BETWEEN 25 AND 45
)
"""
q(CHILD_COUNT_SQL)

,COUNT(*)
0,34533


## 2. JOIN — person-years to family income
`persons ⋈ families` on (`year`, `interview` = `fam_interview`). Inner join, with the row count before and after.

In [5]:
q("""
SELECT year, interview, id_1968, person_num, rel_head
FROM persons
WHERE interview = 32 AND year IN (1990, 1991)
ORDER BY year, rel_head
""")

,year,interview,id_1968,person_num,rel_head
0,1990,32,1830,171,10
1,1990,32,1830,2,22
2,1990,32,1830,31,35
3,1991,32,2062,3,10
4,1991,32,2062,174,20
5,1991,32,2062,31,30
6,1991,32,2062,175,33
7,1991,32,2062,176,33


In [6]:
CHILD_JOIN_SQL = """
SELECT COUNT(*)
FROM persons AS p JOIN families AS f ON p.year = f.year AND p.interview = f.fam_interview
WHERE (p.rel_head IN (1, 2) OR p.rel_head IN (10, 20, 22))
    AND age BETWEEN 25 AND 45
"""
q(CHILD_JOIN_SQL)

,COUNT(*)
0,259398


In [7]:
# Join check: rows before == rows after. (year, fam_interview) is unique -> nothing duplicated;
# every person-year has its family record (checked in 02) -> nothing lost.
n_before = q(CHILD_N_SQL).iloc[0, 0]
n_after = q(CHILD_JOIN_SQL).iloc[0, 0]
print(f"child person-years before join: {n_before:,} | after: {n_after:,}")
assert n_before == n_after

# Keep the joined child person-years as a view: one row per child-year with the child's real family income
CHILD_INCOME_VIEW = """
CREATE TEMP VIEW child_income AS
SELECT p.id_1968, p.person_num, p.year, p.age, f.fam_income_real
FROM persons AS p JOIN families AS f ON p.year = f.year AND p.interview = f.fam_interview
WHERE (p.rel_head IN (1, 2) OR p.rel_head IN (10, 20, 22))
    AND p.age BETWEEN 25 AND 45
"""
make_view("child_income", CHILD_INCOME_VIEW)

child person-years before join: 259,398 | after: 259,398


## 3. JOIN through identifiers — child to father (and/or mother)
`individuals` gives each child the father's (`id_1968`, `person_num`). Join that pair back to the father's own
person-years. Use a LEFT JOIN first and count the NULLs: that is the number of children whose father is not in the
study, and it goes in the post's caveats.

In [8]:
# Self-join: the child's pointer columns (father_id_1968, father_person_num) -> the father's own ID columns.
# LEFT JOIN keeps children whose father is not found; their dad.* columns are NULL. Counted for the child sample only.
CHILD_FATHER_JOIN = """
SELECT COUNT(*)                                            AS children,
       SUM(dad.id_1968 IS NULL)                            AS no_father_found,
       SUM(c.father_id_1968 = 0)                           AS father_unknown,
       SUM(dad.id_1968 IS NULL AND c.father_id_1968 <> 0)  AS father_id_not_in_file
FROM individuals AS c
LEFT JOIN individuals AS dad
    ON c.father_id_1968 = dad.id_1968 AND c.father_person_num = dad.person_num
WHERE EXISTS (SELECT 1 FROM child_income AS ci
              WHERE ci.id_1968 = c.id_1968 AND ci.person_num = c.person_num)
"""
q(CHILD_FATHER_JOIN)

,children,no_father_found,father_unknown,father_id_not_in_file
0,34533,24690,15316,9374


In [9]:
# Child's birth year, to know how old the child was in each of the father's survey years
BIRTH_YEAR_VIEW = """
CREATE TEMP VIEW birth AS
SELECT id_1968, person_num, MIN(year - age) AS birth_year
FROM persons
WHERE age > 0
GROUP BY id_1968, person_num
"""
make_view("birth", BIRTH_YEAR_VIEW)

# Father's person-years while the child was 13-17, with the father's family income that year.
# Inner joins from here on: a child without a matched father has no parent income to use.
FATHER_INCOME_VIEW = """
CREATE TEMP VIEW father_income AS
SELECT c.id_1968, c.person_num, c.father_id_1968, c.father_person_num,
       fp.year, fp.year - b.birth_year AS child_age, f.fam_income_real
FROM individuals AS c
JOIN birth AS b    ON b.id_1968 = c.id_1968 AND b.person_num = c.person_num
JOIN persons AS fp ON fp.id_1968 = c.father_id_1968 AND fp.person_num = c.father_person_num
JOIN families AS f ON f.year = fp.year AND f.fam_interview = fp.interview
WHERE fp.year - b.birth_year BETWEEN 13 AND 17
"""
make_view("father_income", FATHER_INCOME_VIEW)
q("""SELECT COUNT(*) AS father_years, COUNT(DISTINCT id_1968 || '-' || person_num) AS children
     FROM father_income""")

,father_years,children
0,52324,16002


## 4. GROUP BY — parent income, single-year and multi-year
Parent income = father's real family income in the years you chose (the professor's suggestion: 2–3 years is enough).
Produce both: one year, and the average over the years. `GROUP BY` the child; `AVG()` the income; `COUNT()` the years actually observed.

In [10]:
# One row per child. Parent income: single year (child age closest to 15) and the average over child ages 13-17.
# Child income: average over ages 30-40, and one year at each of ages 25, 30, ..., 45 (+-1: waves are biennial after 1997).
MATCHED_SQL = """
WITH parent_avg AS (
    SELECT id_1968, person_num, father_id_1968, father_person_num,
           AVG(fam_income_real) AS parent_inc_avg,
           COUNT(*)             AS parent_years
    FROM father_income
    GROUP BY id_1968, person_num
),
parent_single AS (
    SELECT id_1968, person_num, fam_income_real AS parent_inc_single, child_age AS parent_single_child_age
    FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY id_1968, person_num
                                       ORDER BY ABS(child_age - 15), year) AS rn
          FROM father_income)
    WHERE rn = 1
),
child_avg AS (
    SELECT id_1968, person_num,
           AVG(fam_income_real) AS child_inc_30_40,
           COUNT(*)             AS child_years_30_40
    FROM child_income
    WHERE age BETWEEN 30 AND 40
    GROUP BY id_1968, person_num
),
targets(target_age) AS (VALUES (25), (30), (35), (40), (45)),
child_at_age AS (
    SELECT id_1968, person_num, target_age, fam_income_real
    FROM (SELECT ci.*, t.target_age,
                 ROW_NUMBER() OVER (PARTITION BY ci.id_1968, ci.person_num, t.target_age
                                    ORDER BY ABS(ci.age - t.target_age), ci.year) AS rn
          FROM child_income AS ci
          JOIN targets AS t ON ci.age BETWEEN t.target_age - 1 AND t.target_age + 1)
    WHERE rn = 1
),
child_by_age AS (
    SELECT id_1968, person_num,
           MAX(CASE WHEN target_age = 25 THEN fam_income_real END) AS child_inc_age25,
           MAX(CASE WHEN target_age = 30 THEN fam_income_real END) AS child_inc_age30,
           MAX(CASE WHEN target_age = 35 THEN fam_income_real END) AS child_inc_age35,
           MAX(CASE WHEN target_age = 40 THEN fam_income_real END) AS child_inc_age40,
           MAX(CASE WHEN target_age = 45 THEN fam_income_real END) AS child_inc_age45
    FROM child_at_age
    GROUP BY id_1968, person_num
)
SELECT pa.id_1968, pa.person_num, i.sex, b.birth_year,
       pa.father_id_1968, pa.father_person_num,
       pa.parent_years, ps.parent_single_child_age, ps.parent_inc_single, pa.parent_inc_avg,
       ca.child_years_30_40, ca.child_inc_30_40,
       cb.child_inc_age25, cb.child_inc_age30, cb.child_inc_age35, cb.child_inc_age40, cb.child_inc_age45
FROM parent_avg AS pa
JOIN parent_single AS ps  ON ps.id_1968 = pa.id_1968 AND ps.person_num = pa.person_num
JOIN individuals AS i     ON i.id_1968 = pa.id_1968 AND i.person_num = pa.person_num
JOIN birth AS b           ON b.id_1968 = pa.id_1968 AND b.person_num = pa.person_num
JOIN child_by_age AS cb   ON cb.id_1968 = pa.id_1968 AND cb.person_num = pa.person_num
LEFT JOIN child_avg AS ca ON ca.id_1968 = pa.id_1968 AND ca.person_num = pa.person_num
"""
matched = q(MATCHED_SQL)
assert not matched.duplicated(["id_1968", "person_num"]).any()
matched.to_csv(PROC / "matched.csv", index=False)
print("matched children:", len(matched))
matched["parent_years"].value_counts().sort_index()

matched children: 7622


parent_years
1     441
2    1197
3    1472
4     623
5    3889
Name: count, dtype: int64

## 5. The IGE
$$\ln y_{\text{child}} = \alpha + \beta \ln y_{\text{parent}} + \varepsilon$$
$\beta$ is the intergenerational elasticity. Estimate it (a) with single-year parent income, (b) with the multi-year average,
and (c) separately by the child's age at measurement (e.g. 25, 30, 35, 40, 45) — that last one is the life-cycle bias figure.
Save every estimate with its standard error and N to `results/tables/`.

In [11]:
# Main sample: the SRC (representative) sample, id_1968 1-2930; the SEO low-income oversample (5001-6872)
# is added only as a robustness row, since there are no weights in the extract.
# At least 2 years of parent income and positive incomes (logs). The same children enter the single-year and the
# average regression, so the gap between them is measurement, not a different sample.
matched["father_key"] = matched["father_id_1968"] * 1000 + matched["father_person_num"]
matched["sample"] = np.where(matched["id_1968"] <= 2930, "SRC", "SEO")

def ige(df, y, x):
    """OLS of ln y on ln x; SEs clustered by father (siblings share a father)."""
    fit = sm.OLS(np.log(df[y]), sm.add_constant(np.log(df[x]))).fit(
        cov_type="cluster", cov_kwds={"groups": df["father_key"]})
    return {"beta": fit.params.iloc[1], "se": fit.bse.iloc[1], "n": int(fit.nobs),
            "n_fathers": df["father_key"].nunique()}

base = matched[(matched["parent_years"] >= 2)
               & (matched["parent_inc_single"] > 0) & (matched["parent_inc_avg"] > 0)]

rows = []
for sample_name, keep in {"SRC": ["SRC"], "SRC+SEO": ["SRC", "SEO"]}.items():
    d = base[base["sample"].isin(keep) & (base["child_inc_30_40"] > 0)]
    for spec, x in {"single year (child age ~15)": "parent_inc_single",
                    "average, child ages 13-17": "parent_inc_avg"}.items():
        rows.append({"sample": sample_name, "parent_income": spec,
                     "child_income": "average, ages 30-40", **ige(d, "child_inc_30_40", x)})
ige_estimates = pd.DataFrame(rows)
ige_estimates.to_csv(TAB / "ige_estimates.csv", index=False)
ige_estimates.round(3)

,sample,parent_income,child_income,beta,se,n,n_fathers
0,SRC,single year (child age ~15),"average, ages 30-40",0.249,0.032,3951,2009
1,SRC,"average, child ages 13-17","average, ages 30-40",0.434,0.024,3951,2009
2,SRC+SEO,single year (child age ~15),"average, ages 30-40",0.328,0.027,6207,3055
3,SRC+SEO,"average, child ages 13-17","average, ages 30-40",0.499,0.019,6207,3055


In [12]:
# Life-cycle bias: child income measured at a single age, parent income = multi-year average, SRC sample
src = base[base["sample"] == "SRC"]
rows = []
for age in (25, 30, 35, 40, 45):
    y = f"child_inc_age{age}"
    rows.append({"child_age": age, **ige(src[src[y] > 0], y, "parent_inc_avg")})
ige_by_age = pd.DataFrame(rows)
ige_by_age.to_csv(TAB / "ige_by_child_age.csv", index=False)
ige_by_age.round(3)

,child_age,beta,se,n,n_fathers
0,25,0.249,0.026,3753,2060
1,30,0.355,0.026,3864,1998
2,35,0.452,0.037,3309,1761
3,40,0.524,0.043,2635,1449
4,45,0.450,0.044,1881,1095


## Decisions and caveats (for the post)
- **Child**: head or spouse of their own family unit, ages 25–45; child income = real family income (head + spouse), 2023 dollars.
- **Father match**: self-join of `individuals` on the father's (`id_1968`, `person_num`). Most child-sample members have no
  father in the data (§3 table): many are original 1968 adults whose parents were never in the study, others have an
  unknown father (`father_id_1968 = 0`). Children with a matched father are not a random subset — a selection caveat.
- **Parent income**: father's family income while the child was 13–17; single year = the year the child was closest to 15;
  multi-year = average over the observed years (≥ 2 required, so both specs use the same children).
- **Sample**: SRC only for the main estimates (no weights in the extract); SRC+SEO as a robustness row. Non-positive incomes dropped before logs.
- **Standard errors** clustered by father, since siblings share a father.
- Child's age in the father's years uses birth year = min(year − age), which can be off by one.
- The by-age estimates use different children at each age (younger cohorts are not yet 45), so part of the age profile is composition.